[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abpandey-dev/learn-python-from-scratch-to-advanced/blob/main/notebooks/06-dsa-interview/02_two-pointers.ipynb)

*DSA Interview Handbook · Notebook 2 of 14*

# Two Pointers

Two indices walking toward each other or in the same direction to replace a nested loop.

### 👩‍🏫 From your teacher

**The picture to keep in your head:** Two fingers on a sorted list, moving toward each other: each step rules out many possibilities at once.

**What interviewers really test here:** Recognising sorted input, and proving the pointer moves are safe.

### How to use this notebook

1. **Run the setup cell** just below.
2. **Read each lesson slowly.** Work the numbers on paper before reading on.
3. **Run and then change the examples.** Breaking code on purpose is how you learn.
4. **Try each practice question before opening an answer.** If you're stuck, open one hint at a time.
5. **Say your interview answers out loud.** Knowing something and explaining it are different skills.

### Your roadmap

- [ ] 1. Why two pointers beat two loops *(Basics)*
- [ ] 2. Opposite ends on a sorted array *(Core)*
- [ ] 3. Read and write pointers (in place) *(Core)*
- [ ] 4. One pointer per sequence *(Core)*
- [ ] 5. Harder two-pointer problems *(Advanced)*
- [ ] 20 practice questions
- [ ] Interview questions, out loud

In [1]:
# ▶ Run this cell first. It prepares the notebook; you don't need to read it.
import contextlib, copy, io, math, sys
import numpy as np, matplotlib.pyplot as plt
%matplotlib inline

PLAIN = False

class ListNode:
    def __init__(self, val=0, next=None):
        self.val, self.next = val, next

class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def mk_list(vals, pos=-1):
    dummy = cur = ListNode()
    nodes = []
    for v in vals:
        cur.next = ListNode(v)
        cur = cur.next
        nodes.append(cur)
    if pos >= 0 and nodes:
        nodes[-1].next = nodes[pos]
    return dummy.next

def list_arr(head):
    out, seen = [], set()
    while head and id(head) not in seen:
        seen.add(id(head))
        out.append(head.val)
        head = head.next
    return out

def mk_tree(vals):
    if not vals or vals[0] is None:
        return None
    root = TreeNode(vals[0])
    queue, i = [root], 1
    while queue and i < len(vals):
        node = queue.pop(0)
        for side in ("left", "right"):
            if i < len(vals) and vals[i] is not None:
                child = TreeNode(vals[i])
                setattr(node, side, child)
                queue.append(child)
            i += 1
    return root

def tree_arr(root):
    out, queue = [], [root]
    while queue:
        node = queue.pop(0)
        if node:
            out.append(node.val)
            queue += [node.left, node.right]
        else:
            out.append(None)
    while out and out[-1] is None:
        out.pop()
    return out

def find(root, val):
    if not root or root.val == val:
        return root
    return find(root.left, val) or find(root.right, val)

ListNode.__repr__ = lambda self: f"ListNode{list_arr(self)}"
TreeNode.__repr__ = lambda self: f"TreeNode{tree_arr(self)}"

def plain(x, digits=4):
    """NumPy / pandas results as plain Python values, so they compare with a test's expected answer."""
    np = sys.modules.get("numpy")
    pd = sys.modules.get("pandas")
    if pd is not None:
        if isinstance(x, pd.DataFrame):
            return {str(c): plain(x[c].tolist(), digits) for c in x.columns}
        if isinstance(x, pd.Series):
            return {plain(k, digits): plain(v, digits) for k, v in x.items()}
        if isinstance(x, pd.Index):
            return plain(x.tolist(), digits)
        if x is pd.NaT:
            return None
        if isinstance(x, pd.Timestamp):
            return str(x.date()) if x == x.normalize() else str(x)
    if np is not None:
        if isinstance(x, np.ndarray):
            return plain(x.tolist(), digits)
        if isinstance(x, np.generic):
            return plain(x.item(), digits)
    if isinstance(x, float):
        return None if x != x else round(x, digits) + 0.0
    if isinstance(x, list):
        return [plain(v, digits) for v in x]
    if isinstance(x, tuple):
        return tuple(plain(v, digits) for v in x)
    if isinstance(x, dict):
        return {plain(k, digits): plain(v, digits) for k, v in x.items()}
    if isinstance(x, (set, frozenset)):
        return {plain(v, digits) for v in x}
    return x


def check(fn, tests, plain_compare=None, listify=False, design=False, prep=None, post=None,
          norm=None, verify=None, runner=None, inplace=None):
    """Run your code against the tests and show which pass."""
    use_plain = PLAIN if plain_compare is None else plain_compare
    passed, problems = 0, []
    for args, want in tests:
        try:
            a = copy.deepcopy(args)
            if design:
                obj = fn(*a[0])
                got = [getattr(obj, m)(*p) for m, p in a[1]]
            elif runner is not None:
                got = runner(fn, a)
            else:
                if prep is not None:
                    a = prep(a)
                with contextlib.redirect_stdout(io.StringIO()):
                    got = fn(*a)
                if inplace is not None:
                    got = a[inplace]
                if listify:
                    got = list(got)
                if post is not None:
                    got = post(got)
            if verify is not None:
                ok = bool(verify(copy.deepcopy(args), got))
            elif norm is not None:
                ok = norm(copy.deepcopy(got)) == norm(copy.deepcopy(want))
            else:
                ok = (plain(got), plain(want)) if use_plain else (got, want)
                ok = ok[0] == ok[1]
            ok = ok if isinstance(ok, bool) else False
        except Exception as e:  # your code raised an error: report it, don't stop the notebook
            ok, got = False, f"{type(e).__name__}: {e}"
        passed += ok
        if not ok:
            problems.append((args, want, got))
    plt.close("all")
    if passed == len(tests):
        print(f"✅ All {len(tests)} tests passed. Well done! Now compare with the answers below.")
    else:
        print(f"❌ {passed}/{len(tests)} tests passed. First one to fix:")
        args, want, got = problems[0]
        print("   input:   ", repr(args)[:300])
        print("   expected:", repr(want)[:300])
        print("   you got: ", repr(got)[:300])
    return None

print("Ready! Scroll down and start learning.")

Ready! Scroll down and start learning.


---
# Part 1: The theory in one place

Read this once for the big picture. It's fine if it doesn't all stick yet: the lessons will make it concrete, and you can come back to revise.

### Definition

The **two-pointer technique** is an algorithm design method that uses two indices (pointers) moving through a sequence — toward each other, in the same direction, or over two different sequences — to solve a problem in a single pass. It is not a data structure but a **traversal strategy**.



Its purpose is to reduce problems that naively examine all pairs, O(n²), to O(n) (or O(n log n) including a sort), with O(1) extra space.

### Types of two-pointer traversal

| Type | Start | Movement | Typical use |
| --- | --- | --- | --- |
| **Opposite direction (converging)** | `l = 0`, `r = n − 1` | move inward until they meet | pair sum in sorted array, palindromes, reversing, container with most water |
| **Same direction (fast/slow, read/write)** | both at 0 | reader always advances; writer advances on a kept element | remove duplicates, remove element, partitioning |
| **Two sequences** | one pointer at the start of each | advance the pointer at the smaller element | merging sorted arrays, intersection, subsequence check |
| **Fixed loop + inner pair** | outer index i, pair l, r after it | converge for each i | 3Sum, 3Sum closest, 4Sum |

### Why it is correct: the elimination argument

For converging pointers on a sorted array searching for `a[l] + a[r] = target`:




- If `a[l] + a[r] < target`: for every `k ≤ r`, `a[l] + a[k] ≤ a[l] + a[r] < target`. So index `l` cannot be in any valid pair with the remaining elements → **safe to discard l**.
- If the sum is greater, the symmetric argument discards `r`.



Each step permanently removes one index from consideration, so at most n − 1 steps happen. This is called a **loop invariant**: "if a solution exists, it lies within [l, r]". The invariant holds at the start and after every step, so the algorithm never misses the answer.



For **container with most water**, the area is limited by the shorter line; moving the taller line inward can only shrink the width while the height stays capped, so the shorter line is discarded.

### Complexity

| Problem | Brute force | Two pointers | Space |
| --- | --- | --- | --- |
| Pair with sum (sorted) | O(n²) | O(n) | O(1) |
| Pair with sum (unsorted) | O(n²) | O(n log n) sort + O(n) | O(1) (or hash map O(n) time) |
| 3Sum | O(n³) | O(n²) | O(1) extra |
| k-Sum | O(nᵏ) | O(nᵏ⁻¹) | O(k) recursion |
| Merge two sorted arrays | O((m+n) log(m+n)) | O(m + n) | O(1) if merged in place from the back |
| In-place removal | O(n) with extra list | O(n) | O(1) |

### Prerequisites, advantages and limitations

**👍 Pros**

##### Advantages

- Linear time instead of quadratic
- Constant extra space — works in place
- Short, easy-to-verify code



**👎 Cons**

##### Limitations

- Converging pointers need **sorted** data (or a similar monotonic property)
- Sorting loses original indices — use a hash map if indices are required
- Doesn't directly handle "count all pairs" with many duplicates without care

### Applications

- Merging step of **merge sort**; merging sorted files or database results.
- **Partitioning** in quicksort (Hoare / Lomuto schemes) and the Dutch national flag problem.
- Palindrome checking, string reversal, removing characters in place.
- Finding pairs/triples with a target sum in sorted data (finance, recommendation matching).
- Fast/slow pointers on linked lists (cycle detection, middle element).

---
# Part 2: Lessons, step by step

## Lesson 1: Why two pointers beat two loops

*Level: Basics*

The obvious way to look at pairs is two nested loops: for every `i`, try every `j`. With n = 10,000 that's 50 million pairs — **O(n²)**.



Two pointers keep two indices and move **one** of them each step, based on what they see. Each move throws away many pairs at once, so the total work is about n steps — **O(n)**.



```text
brute force : (0,1) (0,2) (0,3) … (1,2) (1,3) …   ~ n²/2 checks
two pointers: l →            ← r                   ~ n checks
```



The simplest example is checking a palindrome: compare the first and last character, then move both inward.



```python
l, r = 0, len(s) - 1
while l < r:
    if s[l] != s[r]:
        return False
    l, r = l + 1, r - 1
return True
```

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [2]:
def palindrome_lesson():
    s = "racecar"
    l, r = 0, len(s) - 1
    while l < r:
        if s[l] != s[r]:
            return False
        l, r = l + 1, r - 1
    return True


palindrome_lesson()

True

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. s[0] = 'r' and s[6] = 'r' match
2. s[1] = 'a' and s[5] = 'a' match
3. s[2] = 'c' and s[4] = 'c' match
4. Pointers met in the middle → palindrome

</details>


**✏️ Your turn:** you now have what you need for [Q1](#q1), [Q5](#q5), [Q12](#q12). Try one now, or carry on and do them in Part 3.

## Lesson 2: Opposite ends on a sorted array

*Level: Core*

When the array is **sorted**, put `l` at the smallest value and `r` at the largest, and look at `a[l] + a[r]`:




- **Too small?** `a[l]` paired with anything is too small too (everything else is ≤ a[r]). So `a[l]` is useless — move `l` right.
- **Too big?** By the same logic `a[r]` is useless — move `r` left.
- **Equal?** Found it.



That argument — "this element can never be part of an answer, so drop it" — is what makes two pointers correct. Keep it in mind; interviewers ask "why does this work?".



**Container with most water** uses the same shape with a different rule: the shorter wall limits the area, and moving the taller wall can never help, so always move the shorter one.



**3Sum**: sort, fix one number with a loop, and run the pair search on the rest. Skip equal neighbours to avoid duplicate triples. O(n²) instead of O(n³).

**✏️ Your turn:** you now have what you need for [Q13](#q13), [Q15](#q15), [Q14](#q14), [Q17](#q17), [Q4](#q4), [Q18](#q18). Try one now, or carry on and do them in Part 3.

## Lesson 3: Read and write pointers (in place)

*Level: Core*

Both pointers move in the **same direction**. The reader `r` visits every element; the writer `w` marks where the next element you want to keep goes.



```python
w = 0
for r in range(len(nums)):
    if keep(nums[r]):
        nums[w] = nums[r]
        w += 1
return w            # first w slots hold the kept elements
```



This removes items or duplicates in place using O(1) extra memory. Elements before `w` are always "done".

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [3]:
def read_write_lesson():
    nums, val = [3, 2, 2, 3, 4, 3, 5], 3
    w = 0
    for r in range(len(nums)):
        if nums[r] != val:
            nums[w] = nums[r]
            w += 1
        else:
            pass
    return w


read_write_lesson()

4

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Skip 3
2. Keep 2 → copy to slot 0
3. Keep 2 → copy to slot 1
4. Skip 3
5. Keep 4 → copy to slot 2
6. Skip 3
7. Keep 5 → copy to slot 3

</details>


**✏️ Your turn:** you now have what you need for [Q2](#q2), [Q8](#q8), [Q10](#q10). Try one now, or carry on and do them in Part 3.

## Lesson 4: One pointer per sequence

*Level: Core*

With two sorted lists (or strings), give each its own pointer and always advance the one that is "behind". This is the **merge** step of merge sort.



```python
i = j = 0
out = []
while i < len(a) and j < len(b):
    if a[i] <= b[j]:
        out.append(a[i]); i += 1
    else:
        out.append(b[j]); j += 1
out += a[i:] + b[j:]
```



Variations: **is-subsequence** (advance in s only on a match), **intersection** (advance both on equal), and merging **from the back** when the first array has spare room at its end.

**▶ See it run.** Before you run this, predict what will happen. Then change a number and run it again.

In [4]:
def merge_lesson():
    a, b = [1, 4, 7], [2, 3, 8, 9]
    i = j = 0
    out = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            out.append(a[i])
            i += 1
        else:
            out.append(b[j])
            j += 1
    out += a[i:] + b[j:]
    return out


merge_lesson()

[1, 2, 3, 4, 7, 8, 9]

<details>
<summary><b>🔎 What happens, step by step</b></summary>

1. Take the smaller head → 1
2. Take the smaller head → 2
3. Take the smaller head → 3
4. Take the smaller head → 4
5. Take the smaller head → 7
6. One list is empty → append the rest of the other

</details>


**✏️ Your turn:** you now have what you need for [Q3](#q3), [Q7](#q7), [Q11](#q11), [Q9](#q9). Try one now, or carry on and do them in Part 3.

## Lesson 5: Harder two-pointer problems

*Level: Advanced*

#### k-Sum



4Sum = two nested loops + the two-pointer pair search, O(n³). In general k-Sum costs O(n^(k−1)). Always sort first and skip duplicates at **every** level.



#### Trapping rain water



Water above bar i = `min(tallest on left, tallest on right) − height[i]`. With two pointers, keep `lmax` and `rmax`. Whichever side has the smaller max is the limiting side, so its water is already known — compute it and move that pointer. O(n) time, O(1) space.



#### When two pointers is the wrong tool




- The array is not sorted and you can't sort it (you need original indices) → use a hash map.
- You need **contiguous** ranges with a condition → that's a sliding window (next topic), a close cousin.

**✏️ Your turn:** you now have what you need for [Q16](#q16), [Q20](#q20), [Q6](#q6), [Q19](#q19). Try one now, or carry on and do them in Part 3.

---
# Part 3: Practice like an interview

There are 20 questions here, easiest first.

**Work the way you would in a real interview:** restate the problem, try an example by hand, say the simple approach and its cost, improve it, then write the code. Test it with the cell. Only open an answer once you've made a real attempt, because struggling is where the learning happens.

<a id="q1"></a>
### Q1. Valid Palindrome

🟢 Warm-up · *Opposite ends*

After lower-casing and dropping every non-alphanumeric character, does `s` read the same forwards and backwards?

**Examples**

- `s = 'A man, a plan, a canal: Panama'` → `True`
- `s = 'race a car'` → `False`
- `s = ' '` → `True`

In [ ]:
def is_palindrome(s):
    """
    After lower-casing and dropping every non-alphanumeric character, does s
    read the same forwards and backwards?
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_palindrome, [
    (('A man, a plan, a canal: Panama',), True),
    (('race a car',), False),
    ((' ',), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build the cleaned string and compare it with its reverse.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two pointers from both ends skip non-alphanumeric characters and compare letters case-insensitively — no extra string needed.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Build the cleaned string and compare it with its reverse.

```python
def is_palindrome(s):
    clean = [c.lower() for c in s if c.isalnum()]
    return clean == clean[::-1]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Two pointers from both ends skip non-alphanumeric characters and compare letters case-insensitively — no extra string needed.

```python
def is_palindrome(s):
    l, r = 0, len(s) - 1
    while l < r:
        while l < r and not s[l].isalnum():
            l += 1
        while l < r and not s[r].isalnum():
            r -= 1
        if s[l].lower() != s[r].lower():
            return False
        l, r = l + 1, r - 1
    return True
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build the cleaned string and compare it with its reverse. That costs O(n) time and O(n) space."

"I can do better: Two pointers from both ends skip non-alphanumeric characters and compare letters case-insensitively — no extra string needed. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'Ab,c;bA'`

1. Compare 'A' and 'A'
2. Compare 'b' and 'b'
3. Compare 'c' and 'c'
4. Pointers met — it is a palindrome

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/valid-palindrome)

<a id="q2"></a>
### Q2. Remove Duplicates from Sorted Array

🟢 Warm-up · *Read / write*

Remove duplicates from sorted `nums` in place so each value appears once, and return the count k of unique values (the first k slots hold them).

**Examples**

- `nums = [1, 1, 2]` → `2`
- `nums = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4]` → `5`

In [ ]:
def remove_duplicates(nums):
    """
    Remove duplicates from sorted nums in place so each value appears once, and
    return the count k of unique values (the first k slots hold them).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(remove_duplicates, [
    (([1, 1, 2],), 2),
    (([0, 0, 1, 1, 1, 2, 2, 3, 3, 4],), 5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Copy unique values into a new list, write them back, return its length.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

w points at the last unique value written. When nums[r] differs from nums[w], it's new: write it at w + 1.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Copy unique values into a new list, write them back, return its length.

```python
def remove_duplicates(nums):
    uniq = []
    for x in nums:
        if not uniq or uniq[-1] != x:
            uniq.append(x)
    nums[:len(uniq)] = uniq
    return len(uniq)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** w points at the last unique value written. When nums[r] differs from nums[w], it's new: write it at w + 1.

```python
def remove_duplicates(nums):
    w = 0
    for r in range(1, len(nums)):
        if nums[r] != nums[w]:
            w += 1
            nums[w] = nums[r]
    return w + 1
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Copy unique values into a new list, write them back, return its length. That costs O(n) time and O(n) space."

"I can do better: w points at the last unique value written. When nums[r] differs from nums[w], it's new: write it at w + 1. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [0, 0, 1, 1, 1, 2, 2, 3]`

1. r = 1: nums[1] = 0. Unique values so far: 1
2. r = 2: nums[2] = 1. Unique values so far: 2
3. r = 3: nums[3] = 1. Unique values so far: 2
4. r = 4: nums[4] = 1. Unique values so far: 2
5. r = 5: nums[5] = 2. Unique values so far: 3
6. r = 6: nums[6] = 2. Unique values so far: 3
7. r = 7: nums[7] = 3. Unique values so far: 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/remove-duplicates-from-sorted-array)

<a id="q3"></a>
### Q3. Merge Sorted Array

🟢 Warm-up · *Fill from the back*

Merge sorted `nums2` (n items) into sorted `nums1`, which has m items followed by n spare zeros. Modify nums1 in place.

**Examples**

- `nums1 = [1, 2, 3, 0, 0, 0], m = 3, nums2 = [2, 5, 6], n = 3` → `[1, 2, 2, 3, 5, 6]`
- `nums1 = [1], m = 1, nums2 = [], n = 0` → `[1]`
- `nums1 = [0], m = 0, nums2 = [1], n = 1` → `[1]`

In [ ]:
def merge(nums1, m, nums2, n):
    """
    Merge sorted nums2 (n items) into sorted nums1, which has m items followed
    by n spare zeros. Modify nums1 in place.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(merge, [(([1, 2, 3, 0, 0, 0], 3, [2, 5, 6], 3), [1, 2, 2, 3, 5, 6]), (([1], 1, [], 0), [1]), (([0], 0, [1], 1), [1])], inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Copy nums2 into the spare slots and sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Fill nums1 from the end: compare the largest remaining of each array and place the bigger at position k. Nothing useful gets overwritten.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Copy nums2 into the spare slots and sort.

```python
def merge(nums1, m, nums2, n):
    nums1[m:] = nums2
    nums1.sort()
```

**Cost:** O((m+n) log(m+n)) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Fill nums1 from the end: compare the largest remaining of each array and place the bigger at position k. Nothing useful gets overwritten.

```python
def merge(nums1, m, nums2, n):
    i, j, k = m - 1, n - 1, m + n - 1
    while j >= 0:
        if i >= 0 and nums1[i] > nums2[j]:
            nums1[k] = nums1[i]
            i -= 1
        else:
            nums1[k] = nums2[j]
            j -= 1
        k -= 1
```

**Cost:** O(m + n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Copy nums2 into the spare slots and sort. That costs O((m+n) log(m+n)) time and O(1) space."

"I can do better: Fill nums1 from the end: compare the largest remaining of each array and place the bigger at position k. Nothing useful gets overwritten. That's O(m + n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums1 = [1, 2, 3, 0, 0, 0], m = 3, nums2 = [2, 5, 6], n = 3`

1. Place the larger tail value at k = 5
2. Place the larger tail value at k = 4
3. Place the larger tail value at k = 3
4. Place the larger tail value at k = 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/merge-sorted-array)

<a id="q4"></a>
### Q4. Squares of a Sorted Array

🟢 Warm-up · *Opposite ends*

Given sorted `nums` (may contain negatives), return the squares of each number, sorted.

**Examples**

- `nums = [-4, -1, 0, 3, 10]` → `[0, 1, 9, 16, 100]`
- `nums = [-7, -3, 2, 3, 11]` → `[4, 9, 9, 49, 121]`

In [ ]:
def sorted_squares(nums):
    """
    Given sorted nums (may contain negatives), return the squares of each
    number, sorted.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sorted_squares, [
    (([-4, -1, 0, 3, 10],), [0, 1, 9, 16, 100]),
    (([-7, -3, 2, 3, 11],), [4, 9, 9, 49, 121]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Square everything, then sort.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

The largest square is at one of the two ends. Compare |nums[l]| and |nums[r]|, write the bigger square at the back of the result, and move that pointer.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Square everything, then sort.

```python
def sorted_squares(nums):
    return sorted(x * x for x in nums)
```

**Cost:** O(n log n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** The largest square is at one of the two ends. Compare |nums[l]| and |nums[r]|, write the bigger square at the back of the result, and move that pointer.

```python
def sorted_squares(nums):
    n = len(nums)
    res = [0] * n
    l, r = 0, n - 1
    for k in range(n - 1, -1, -1):
        if abs(nums[l]) > abs(nums[r]):
            res[k] = nums[l] ** 2
            l += 1
        else:
            res[k] = nums[r] ** 2
            r -= 1
    return res
```

**Cost:** O(n) time, O(n) output space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Square everything, then sort. That costs O(n log n) time and O(n) space."

"I can do better: The largest square is at one of the two ends. Compare |nums[l]| and |nums[r]|, write the bigger square at the back of the result, and move that pointer. That's O(n) time and O(n) output space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [-4, -1, 0, 3, 10]`

1. Bigger square goes to res[4]
2. Bigger square goes to res[3]
3. Bigger square goes to res[2]
4. Bigger square goes to res[1]
5. Bigger square goes to res[0]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/squares-of-a-sorted-array)

<a id="q5"></a>
### Q5. Reverse String

🟢 Warm-up · *Opposite ends · swap*

Reverse a list of characters in place.

**Examples**

- `s = ['h', 'e', 'l', 'l', 'o']` → `['o', 'l', 'l', 'e', 'h']`
- `s = ['H', 'a', 'n', 'n', 'a', 'h']` → `['h', 'a', 'n', 'n', 'a', 'H']`

In [ ]:
def reverse_string(s):
    """
    Reverse a list of characters in place.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse_string, [((["h", "e", "l", "l", "o"],), ["o", "l", "l", "e", "h"]), ((["H", "a", "n", "n", "a", "h"],), ["h", "a", "n", "n", "a", "H"])], inplace=0)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a reversed copy and write it back.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Swap s[l] and s[r], then move both pointers inward until they meet.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Build a reversed copy and write it back.

```python
def reverse_string(s):
    s[:] = [s[i] for i in range(len(s) - 1, -1, -1)]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Swap s[l] and s[r], then move both pointers inward until they meet.

```python
def reverse_string(s):
    l, r = 0, len(s) - 1
    while l < r:
        s[l], s[r] = s[r], s[l]
        l, r = l + 1, r - 1
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a reversed copy and write it back. That costs O(n) time and O(n) space."

"I can do better: Swap s[l] and s[r], then move both pointers inward until they meet. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = ['a', 'b', 'c', 'd', 'e', 'f']`

1. Swap positions 0 and 5
2. Swap positions 1 and 4
3. Swap positions 2 and 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/reverse-string)

<a id="q6"></a>
### Q6. Valid Palindrome II

🟢 Warm-up · *Opposite ends · one skip*

Return `True` if `s` can become a palindrome after deleting at most one character.

**Examples**

- `s = 'aba'` → `True`
- `s = 'abca'` → `True`
- `s = 'abc'` → `False`

In [ ]:
def valid_palindrome(s):
    """
    Return True if s can become a palindrome after deleting at most one
    character.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(valid_palindrome, [
    (('aba',), True),
    (('abca',), True),
    (('abc',), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try deleting each character and check if the rest is a palindrome.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Compare from both ends. At the first mismatch, the only options are skipping s[l] or skipping s[r] — check each remaining middle once.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try deleting each character and check if the rest is a palindrome.

```python
def valid_palindrome(s):
    if s == s[::-1]:
        return True
    for i in range(len(s)):
        t = s[:i] + s[i + 1:]
        if t == t[::-1]:
            return True
    return False
```

**Cost:** O(n²) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Compare from both ends. At the first mismatch, the only options are skipping s[l] or skipping s[r] — check each remaining middle once.

```python
def valid_palindrome(s):
    def is_pal(l, r):
        while l < r:
            if s[l] != s[r]:
                return False
            l, r = l + 1, r - 1
        return True

    l, r = 0, len(s) - 1
    while l < r:
        if s[l] != s[r]:
            return is_pal(l + 1, r) or is_pal(l, r - 1)
        l, r = l + 1, r - 1
    return True
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try deleting each character and check if the rest is a palindrome. That costs O(n²) time and O(n) space."

"I can do better: Compare from both ends. At the first mismatch, the only options are skipping s[l] or skipping s[r] — check each remaining middle once. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'racecbar'`

1. Compare 'r' and 'r'
2. Compare 'a' and 'a'
3. Compare 'c' and 'b'
4. Mismatch → try skipping s[2] or s[5]

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/valid-palindrome-ii)

<a id="q7"></a>
### Q7. Is Subsequence

🟢 Warm-up · *Two sequences*

Return `True` if `s` can be formed by deleting some characters of `t` without reordering.

**Examples**

- `s = 'abc', t = 'ahbgdc'` → `True`
- `s = 'axc', t = 'ahbgdc'` → `False`
- `s = '', t = 'abc'` → `True`

In [ ]:
def is_subsequence(s, t):
    """
    Return True if s can be formed by deleting some characters of t without
    reordering.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(is_subsequence, [
    (('abc', 'ahbgdc'), True),
    (('axc', 'ahbgdc'), False),
    (('', 'abc'), True),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every subset of positions of t (2^|t| options).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Walk t with pointer j; advance i in s whenever the characters match. s is a subsequence if i reaches the end.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every subset of positions of t (2^|t| options).

```python
def is_subsequence(s, t):
    from itertools import combinations
    return any("".join(c) == s for c in combinations(t, len(s)))
```

**Cost:** O(2^m · n) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Walk t with pointer j; advance i in s whenever the characters match. s is a subsequence if i reaches the end.

```python
def is_subsequence(s, t):
    i = 0
    for j, ch in enumerate(t):
        if i < len(s) and s[i] == ch:
            i += 1
        else:
            pass
    return i == len(s)
```

**Cost:** O(m) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every subset of positions of t (2^|t| options). That costs O(2^m · n) time and O(m) space."

"I can do better: Walk t with pointer j; advance i in s whenever the characters match. s is a subsequence if i reaches the end. That's O(m) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'abc', t = 'ahbgdc'`

1. t[0] = 'a' matches s[0] → advance i
2. t[1] = 'h' doesn't match
3. t[2] = 'b' matches s[1] → advance i
4. t[3] = 'g' doesn't match
5. t[4] = 'd' doesn't match
6. t[5] = 'c' matches s[2] → advance i

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/is-subsequence)

<a id="q8"></a>
### Q8. Remove Element

🟢 Warm-up · *Read / write*

Remove every occurrence of `val` from `nums` in place and return how many elements remain.

**Examples**

- `nums = [3, 2, 2, 3], val = 3` → `2`
- `nums = [0, 1, 2, 2, 3, 0, 4, 2], val = 2` → `5`

In [ ]:
def remove_element(nums, val):
    """
    Remove every occurrence of val from nums in place and return how many
    elements remain.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(remove_element, [
    (([3, 2, 2, 3], 3), 2),
    (([0, 1, 2, 2, 3, 0, 4, 2], 2), 5),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Build a filtered copy and write it back.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Write pointer w: copy every element that is not val to nums[w] and advance w.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Build a filtered copy and write it back.

```python
def remove_element(nums, val):
    keep = [x for x in nums if x != val]
    nums[:len(keep)] = keep
    return len(keep)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Write pointer w: copy every element that is not val to nums[w] and advance w.

```python
def remove_element(nums, val):
    w = 0
    for r in range(len(nums)):
        if nums[r] != val:
            nums[w] = nums[r]
            w += 1
    return w
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Build a filtered copy and write it back. That costs O(n) time and O(n) space."

"I can do better: Write pointer w: copy every element that is not val to nums[w] and advance w. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [3, 2, 2, 3], val = 3`

1. nums[0] = 3 = 3
2. nums[1] = 2 ≠ 3
3. nums[2] = 2 ≠ 3
4. nums[3] = 3 = 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/remove-element)

<a id="q9"></a>
### Q9. Backspace String Compare

🟢 Warm-up · *Two pointers from the back*

`#` means backspace. Return `True` if `s` and `t` are equal after typing them into empty editors.

**Examples**

- `s = 'ab#c', t = 'ad#c'` → `True`
- `s = 'ab##', t = 'c#d#'` → `True`
- `s = 'a#c', t = 'b'` → `False`

In [ ]:
def backspace_compare(s, t):
    """
    # means backspace. Return True if s and t are equal after typing them into
    empty editors.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(backspace_compare, [
    (('ab#c', 'ad#c'), True),
    (('ab##', 'c#d#'), True),
    (('a#c', 'b'), False),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Simulate both strings with a stack and compare.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Walk both strings from the end, counting pending backspaces to skip characters; compare the next real characters.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Simulate both strings with a stack and compare.

```python
def backspace_compare(s, t):
    def build(x):
        st = []
        for c in x:
            if c == "#":
                if st:
                    st.pop()
            else:
                st.append(c)
        return st
    return build(s) == build(t)
```

**Cost:** O(n + m) time, O(n + m) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Walk both strings from the end, counting pending backspaces to skip characters; compare the next real characters.

```python
def backspace_compare(s, t):
    def next_char(x, i):
        skip = 0
        while i >= 0:
            if x[i] == "#":
                skip += 1
            elif skip:
                skip -= 1
            else:
                break
            i -= 1
        return i

    i, j = len(s) - 1, len(t) - 1
    while True:
        i, j = next_char(s, i), next_char(t, j)
        if i < 0 or j < 0:
            return i < 0 and j < 0
        if s[i] != t[j]:
            return False
        i, j = i - 1, j - 1
```

**Cost:** O(n + m) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Simulate both strings with a stack and compare. That costs O(n + m) time and O(n + m) space."

"I can do better: Walk both strings from the end, counting pending backspaces to skip characters; compare the next real characters. That's O(n + m) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'ab#c', t = 'ad#c'`

1. Next real characters: c vs c
2. Next real characters: a vs a
3. Next real characters: ∅ vs ∅

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/backspace-string-compare)

<a id="q10"></a>
### Q10. Sort Array By Parity

🟢 Warm-up · *Partition*

Move all even numbers before all odd numbers (any order inside each group).

**Examples**

- `nums = [3, 1, 2, 4]` → `[2, 4, 3, 1]`
- `nums = [0]` → `[0]`

In [ ]:
def sort_array_by_parity(nums):
    """
    Move all even numbers before all odd numbers (any order inside each group).
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(sort_array_by_parity, [(([3, 1, 2, 4],), [2, 4, 3, 1]), (([0],), [0])], verify=lambda a, out: sorted(out) == sorted(a[0]) and all(x % 2 == 0 for x in out[:sum(1 for y in out if y % 2 == 0)]))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Return evens + odds as two new lists.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Partition in place: l looks for an odd from the left, r for an even from the right; swap them.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Return evens + odds as two new lists.

```python
def sort_array_by_parity(nums):
    return [x for x in nums if x % 2 == 0] + [x for x in nums if x % 2]
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Partition in place: l looks for an odd from the left, r for an even from the right; swap them.

```python
def sort_array_by_parity(nums):
    l, r = 0, len(nums) - 1
    while l < r:
        if nums[l] % 2 > nums[r] % 2:
            nums[l], nums[r] = nums[r], nums[l]
        if nums[l] % 2 == 0:
            l += 1
        if nums[r] % 2 == 1:
            r -= 1
    return nums
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Return evens + odds as two new lists. That costs O(n) time and O(n) space."

"I can do better: Partition in place: l looks for an odd from the left, r for an even from the right; swap them. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [3, 1, 2, 4, 7, 6]`

1. Odd on the left, even on the right → swap
2. Advance pointers past correctly placed numbers
3. Advance pointers past correctly placed numbers
4. Odd on the left, even on the right → swap
5. Advance pointers past correctly placed numbers

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/sort-array-by-parity)

<a id="q11"></a>
### Q11. Intersection of Two Arrays II

🟢 Warm-up · *Sort + two sequences*

Return the intersection of two arrays, where each element appears as many times as it shows in both.

**Examples**

- `nums1 = [1, 2, 2, 1], nums2 = [2, 2]` → `[2, 2]`
- `nums1 = [4, 9, 5], nums2 = [9, 4, 9, 8, 4]` → `[4, 9]`

In [ ]:
def intersect(nums1, nums2):
    """
    Return the intersection of two arrays, where each element appears as many
    times as it shows in both.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(intersect, [(([1, 2, 2, 1], [2, 2]), [2, 2]), (([4, 9, 5], [9, 4, 9, 8, 4]), [4, 9])], norm=sorted)

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each element of nums1, search and remove a match in a copy of nums2.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort both, then walk them together: equal → take it and advance both; otherwise advance the smaller.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each element of nums1, search and remove a match in a copy of nums2.

```python
def intersect(nums1, nums2):
    rest, res = list(nums2), []
    for x in nums1:
        if x in rest:
            rest.remove(x)
            res.append(x)
    return res
```

**Cost:** O(n · m) time, O(m) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort both, then walk them together: equal → take it and advance both; otherwise advance the smaller.

```python
def intersect(nums1, nums2):
    nums1.sort()
    nums2.sort()
    i = j = 0
    res = []
    while i < len(nums1) and j < len(nums2):
        if nums1[i] == nums2[j]:
            res.append(nums1[i])
            i, j = i + 1, j + 1
        elif nums1[i] < nums2[j]:
            i += 1
        else:
            j += 1
    return res
```

**Cost:** O(n log n + m log m) time, O(1) extra space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each element of nums1, search and remove a match in a copy of nums2. That costs O(n · m) time and O(m) space."

"I can do better: Sort both, then walk them together: equal → take it and advance both; otherwise advance the smaller. That's O(n log n + m log m) time and O(1) extra space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums1 = [1, 2, 2, 1], nums2 = [2, 2]`

1. 1 vs 2
2. 1 vs 2
3. 2 vs 2
4. 2 vs 2

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/intersection-of-two-arrays-ii)

<a id="q12"></a>
### Q12. Reverse Vowels of a String

🟢 Warm-up · *Opposite ends · swap*

Reverse only the vowels (a, e, i, o, u in either case) of `s`.

**Examples**

- `s = 'IceCreAm'` → `'AceCreIm'`
- `s = 'leetcode'` → `'leotcede'`

In [ ]:
def reverse_vowels(s):
    """
    Reverse only the vowels (a, e, i, o, u in either case) of s.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse_vowels, [
    (('IceCreAm',), 'AceCreIm'),
    (('leetcode',), 'leotcede'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Collect the vowels, then rebuild the string taking vowels from the end of that list.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Two pointers move inward, each stopping at a vowel; swap them.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Collect the vowels, then rebuild the string taking vowels from the end of that list.

```python
def reverse_vowels(s):
    vow = [c for c in s if c in "aeiouAEIOU"]
    return "".join(vow.pop() if c in "aeiouAEIOU" else c for c in s)
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Two pointers move inward, each stopping at a vowel; swap them.

```python
def reverse_vowels(s):
    a = list(s)
    l, r = 0, len(a) - 1
    while l < r:
        while l < r and a[l] not in "aeiouAEIOU":
            l += 1
        while l < r and a[r] not in "aeiouAEIOU":
            r -= 1
        a[l], a[r] = a[r], a[l]
        l, r = l + 1, r - 1
    return "".join(a)
```

**Cost:** O(n) time, O(n) for the list copy, O(1) extra otherwise space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Collect the vowels, then rebuild the string taking vowels from the end of that list. That costs O(n) time and O(n) space."

"I can do better: Two pointers move inward, each stopping at a vowel; swap them. That's O(n) time and O(n) for the list copy, O(1) extra otherwise space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'IceCreAm'`

1. Swap vowels at 0 and 6
2. Swap vowels at 2 and 5
3. Swap vowels at 4 and 4

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/reverse-vowels-of-a-string)

<a id="q13"></a>
### Q13. Two Sum II - Input Array Is Sorted

🟡 Interview standard · *Opposite ends*

`numbers` is sorted ascending. Return the 1-based indices of the two numbers that add up to `target`, using O(1) extra space.

**Examples**

- `numbers = [2, 7, 11, 15], target = 9` → `[1, 2]`
- `numbers = [2, 3, 4], target = 6` → `[1, 3]`
- `numbers = [-1, 0], target = -1` → `[1, 2]`

In [ ]:
def two_sum_sorted(numbers, target):
    """
    numbers is sorted ascending. Return the 1-based indices of the two numbers
    that add up to target, using O(1) extra space.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(two_sum_sorted, [
    (([2, 7, 11, 15], 9), [1, 2]),
    (([2, 3, 4], 6), [1, 3]),
    (([-1, 0], -1), [1, 2]),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every pair.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Start at both ends. Sum too small → move l right; too big → move r left. Sorting guarantees we never skip the answer.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every pair.

```python
def two_sum_sorted(numbers, target):
    for i in range(len(numbers)):
        for j in range(i + 1, len(numbers)):
            if numbers[i] + numbers[j] == target:
                return [i + 1, j + 1]
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Start at both ends. Sum too small → move l right; too big → move r left. Sorting guarantees we never skip the answer.

```python
def two_sum_sorted(numbers, target):
    l, r = 0, len(numbers) - 1
    while l < r:
        total = numbers[l] + numbers[r]
        if total == target:
            return [l + 1, r + 1]
        if total < target:
            l += 1
        else:
            r -= 1
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every pair. That costs O(n²) time and O(1) space."

"I can do better: Start at both ends. Sum too small → move l right; too big → move r left. Sorting guarantees we never skip the answer. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `numbers = [1, 2, 4, 6, 9, 12], target = 8`

1. 1 + 12 = 13
2. 1 + 9 = 10
3. 1 + 6 = 7
4. 2 + 6 = 8

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/two-sum-ii-input-array-is-sorted)

<a id="q14"></a>
### Q14. 3Sum

🟡 Interview standard · *Sort + two pointers*

Return every unique triplet `[a, b, c]` from `nums` with a + b + c = 0.

**Examples**

- `nums = [-1, 0, 1, 2, -1, -4]` → `[[-1, -1, 2], [-1, 0, 1]]`
- `nums = [0, 1, 1]` → `[]`
- `nums = [0, 0, 0]` → `[[0, 0, 0]]`

In [ ]:
def three_sum(nums):
    """
    Return every unique triplet [a, b, c] from nums with a + b + c = 0.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(three_sum, [(([-1, 0, 1, 2, -1, -4],), [[-1, -1, 2], [-1, 0, 1]]), (([0, 1, 1],), []), (([0, 0, 0],), [[0, 0, 0]])], norm=lambda r: sorted(sorted(x) for x in r))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every triple i &lt; j &lt; k and collect sorted triples in a set.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort. Fix nums[i] and find pairs summing to −nums[i] with two pointers on the rest. Skip equal neighbours so each triplet appears once.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every triple i &lt; j &lt; k and collect sorted triples in a set.

```python
def three_sum(nums):
    n, res = len(nums), set()
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                if nums[i] + nums[j] + nums[k] == 0:
                    res.add(tuple(sorted((nums[i], nums[j], nums[k]))))
    return [list(t) for t in res]
```

**Cost:** O(n³) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort. Fix nums[i] and find pairs summing to −nums[i] with two pointers on the rest. Skip equal neighbours so each triplet appears once.

```python
def three_sum(nums):
    nums.sort()
    res = []
    for i in range(len(nums) - 2):
        if i > 0 and nums[i] == nums[i - 1]:
            continue
        l, r = i + 1, len(nums) - 1
        while l < r:
            total = nums[i] + nums[l] + nums[r]
            if total < 0:
                l += 1
            elif total > 0:
                r -= 1
            else:
                res.append([nums[i], nums[l], nums[r]])
                while l < r and nums[l] == nums[l + 1]:
                    l += 1
                l, r = l + 1, r - 1
    return res
```

**Cost:** O(n²) time, O(1) extra space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every triple i &lt; j &lt; k and collect sorted triples in a set. That costs O(n³) time and O(n) space."

"I can do better: Sort. Fix nums[i] and find pairs summing to −nums[i] with two pointers on the rest. Skip equal neighbours so each triplet appears once. That's O(n²) time and O(1) extra space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [-1, 0, 1, 2, -1, -4]`

1. Fix -4: -4 + -1 + 2 = -3
2. Fix -4: -4 + -1 + 2 = -3
3. Fix -4: -4 + 0 + 2 = -2
4. Fix -4: -4 + 1 + 2 = -1
5. Fix -1: -1 + -1 + 2 = 0
6. Fix -1: -1 + 0 + 1 = 0
7. Fix 0: 0 + 1 + 2 = 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/3sum)

<a id="q15"></a>
### Q15. Container With Most Water

🟡 Interview standard · *Opposite ends · greedy*

Line i has height `height[i]`. Pick two lines that, with the x-axis, hold the most water and return that area.

**Examples**

- `height = [1, 8, 6, 2, 5, 4, 8, 3, 7]` → `49`
- `height = [1, 1]` → `1`

In [ ]:
def max_area(height):
    """
    Line i has height height[i]. Pick two lines that, with the x-axis, hold the
    most water and return that area.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(max_area, [
    (([1, 8, 6, 2, 5, 4, 8, 3, 7],), 49),
    (([1, 1],), 1),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every pair of lines.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Start with the widest pair. The shorter line limits the area, and moving the taller one inward can only shrink it — so always move the shorter line.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every pair of lines.

```python
def max_area(height):
    best = 0
    for i in range(len(height)):
        for j in range(i + 1, len(height)):
            best = max(best, (j - i) * min(height[i], height[j]))
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Start with the widest pair. The shorter line limits the area, and moving the taller one inward can only shrink it — so always move the shorter line.

```python
def max_area(height):
    l, r, best = 0, len(height) - 1, 0
    while l < r:
        area = (r - l) * min(height[l], height[r])
        best = max(best, area)
        if height[l] < height[r]:
            l += 1
        else:
            r -= 1
    return best
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every pair of lines. That costs O(n²) time and O(1) space."

"I can do better: Start with the widest pair. The shorter line limits the area, and moving the taller one inward can only shrink it — so always move the shorter line. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `height = [1, 8, 6, 2, 5, 4, 8, 3, 7]`

1. width 8 × min(1, 7) = 8. Move the shorter line.
2. width 7 × min(8, 7) = 49. Move the shorter line.
3. width 6 × min(8, 3) = 18. Move the shorter line.
4. width 5 × min(8, 8) = 40. Move the shorter line.
5. width 4 × min(8, 4) = 16. Move the shorter line.
6. width 3 × min(8, 5) = 15. Move the shorter line.
7. width 2 × min(8, 2) = 4. Move the shorter line.
8. width 1 × min(8, 6) = 6. Move the shorter line.

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/container-with-most-water)

<a id="q16"></a>
### Q16. 4Sum

🟡 Interview standard · *Sort + two loops + two pointers*

Return all unique quadruplets in `nums` that sum to `target`.

**Examples**

- `nums = [1, 0, -1, 0, -2, 2], target = 0` → `[[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]`
- `nums = [2, 2, 2, 2, 2], target = 8` → `[[2, 2, 2, 2]]`

In [ ]:
def four_sum(nums, target):
    """
    Return all unique quadruplets in nums that sum to target.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(four_sum, [(([1, 0, -1, 0, -2, 2], 0), [[-2, -1, 1, 2], [-2, 0, 0, 2], [-1, 0, 0, 1]]), (([2, 2, 2, 2, 2], 8), [[2, 2, 2, 2]])], norm=lambda r: sorted(sorted(x) for x in r))

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every 4-element combination and keep unique sorted ones.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort. Two nested loops fix a and b; two pointers find c + d = target − a − b. Skip duplicates at every level.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every 4-element combination and keep unique sorted ones.

```python
def four_sum(nums, target):
    from itertools import combinations
    return [list(c) for c in {tuple(sorted(c)) for c in combinations(nums, 4) if sum(c) == target}]
```

**Cost:** O(n⁴) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort. Two nested loops fix a and b; two pointers find c + d = target − a − b. Skip duplicates at every level.

```python
def four_sum(nums, target):
    nums.sort()
    n, res = len(nums), []
    for a in range(n - 3):
        if a > 0 and nums[a] == nums[a - 1]:
            continue
        for b in range(a + 1, n - 2):
            if b > a + 1 and nums[b] == nums[b - 1]:
                continue
            l, r = b + 1, n - 1
            while l < r:
                total = nums[a] + nums[b] + nums[l] + nums[r]
                if total < target:
                    l += 1
                elif total > target:
                    r -= 1
                else:
                    res.append([nums[a], nums[b], nums[l], nums[r]])
                    while l < r and nums[l] == nums[l + 1]:
                        l += 1
                    l, r = l + 1, r - 1
    return res
```

**Cost:** O(n³) time, O(1) extra space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every 4-element combination and keep unique sorted ones. That costs O(n⁴) time and O(n) space."

"I can do better: Sort. Two nested loops fix a and b; two pointers find c + d = target − a − b. Skip duplicates at every level. That's O(n³) time and O(1) extra space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [2, -1, 0, 1, -2, 0], target = 0`

1. -2 + -1 + 0 + 2 = -1
2. -2 + -1 + 0 + 2 = -1
3. -2 + -1 + 1 + 2 = 0
4. -2 + 0 + 0 + 2 = 0
5. -1 + 0 + 0 + 2 = 1
6. -1 + 0 + 0 + 1 = 0
7. 0 + 0 + 1 + 2 = 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/4sum)

<a id="q17"></a>
### Q17. 3Sum Closest

🟡 Interview standard · *Sort + two pointers*

Find three integers in `nums` whose sum is closest to `target` and return that sum.

**Examples**

- `nums = [-1, 2, 1, -4], target = 1` → `2`
- `nums = [0, 0, 0], target = 1` → `0`

In [ ]:
def three_sum_closest(nums, target):
    """
    Find three integers in nums whose sum is closest to target and return that
    sum.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(three_sum_closest, [
    (([-1, 2, 1, -4], 1), 2),
    (([0, 0, 0], 1), 0),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every triple and keep the closest sum.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort, fix one element, and move two pointers toward the target sum, tracking the closest seen.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every triple and keep the closest sum.

```python
def three_sum_closest(nums, target):
    best = nums[0] + nums[1] + nums[2]
    n = len(nums)
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                s = nums[i] + nums[j] + nums[k]
                if abs(s - target) < abs(best - target):
                    best = s
    return best
```

**Cost:** O(n³) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort, fix one element, and move two pointers toward the target sum, tracking the closest seen.

```python
def three_sum_closest(nums, target):
    nums.sort()
    best = nums[0] + nums[1] + nums[2]
    for i in range(len(nums) - 2):
        l, r = i + 1, len(nums) - 1
        while l < r:
            s = nums[i] + nums[l] + nums[r]
            if abs(s - target) < abs(best - target):
                best = s
            if s < target:
                l += 1
            elif s > target:
                r -= 1
            else:
                return s
    return best
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every triple and keep the closest sum. That costs O(n³) time and O(1) space."

"I can do better: Sort, fix one element, and move two pointers toward the target sum, tracking the closest seen. That's O(n²) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `nums = [-4, -1, 1, 2, 5], target = 3`

1. sum 0, closest so far 0
2. sum 2, closest so far 2
3. sum 3, closest so far 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/3sum-closest)

<a id="q18"></a>
### Q18. Boats to Save People

🟡 Interview standard · *Sort + greedy pairing*

Each boat carries at most two people and at most `limit` weight. Return the minimum number of boats.

**Examples**

- `people = [1, 2], limit = 3` → `1`
- `people = [3, 2, 2, 1], limit = 3` → `3`
- `people = [3, 5, 3, 4], limit = 5` → `4`

In [ ]:
def num_rescue_boats(people, limit):
    """
    Each boat carries at most two people and at most limit weight. Return the
    minimum number of boats.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(num_rescue_boats, [
    (([1, 2], 3), 1),
    (([3, 2, 2, 1], 3), 3),
    (([3, 5, 3, 4], 5), 4),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Try every way to pair people (exponential).

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Sort. The heaviest person always takes a boat; pair them with the lightest if they fit together. Move the pointers inward.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Try every way to pair people (exponential).

```python
def num_rescue_boats(people, limit):
    from functools import lru_cache
    people = tuple(sorted(people))

    @lru_cache(None)
    def go(rest):
        if not rest:
            return 0
        first, others = rest[0], rest[1:]
        best = 1 + go(others)
        for i in range(len(others)):
            if first + others[i] <= limit:
                best = min(best, 1 + go(others[:i] + others[i + 1:]))
        return best
    return go(people)
```

**Cost:** O(n!) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Sort. The heaviest person always takes a boat; pair them with the lightest if they fit together. Move the pointers inward.

```python
def num_rescue_boats(people, limit):
    people.sort()
    l, r, boats = 0, len(people) - 1, 0
    while l <= r:
        pair = l < r and people[l] + people[r] <= limit
        if people[l] + people[r] <= limit:
            l += 1
        r -= 1
        boats += 1
    return boats
```

**Cost:** O(n log n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Try every way to pair people (exponential). That costs O(n!) time and O(n) space."

"I can do better: Sort. The heaviest person always takes a boat; pair them with the lightest if they fit together. Move the pointers inward. That's O(n log n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `people = [3, 2, 5, 1, 4], limit = 6`

1. Heaviest 5 + lightest 1 fit together
2. Heaviest 4 + lightest 2 fit together
3. Heaviest 3 goes alone

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/boats-to-save-people)

<a id="q19"></a>
### Q19. Reverse Words in a String

🟡 Interview standard · *Reverse twice*

Reverse the order of words in `s`. Words are separated by one or more spaces; return them joined by single spaces.

**Examples**

- `s = 'the sky is blue'` → `'blue is sky the'`
- `s = '  hello world  '` → `'world hello'`
- `s = 'a good   example'` → `'example good a'`

In [ ]:
def reverse_words(s):
    """
    Reverse the order of words in s. Words are separated by one or more spaces;
    return them joined by single spaces.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(reverse_words, [
    (('the sky is blue',), 'blue is sky the'),
    (('  hello world  ',), 'world hello'),
    (('a good   example',), 'example good a'),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

Split on spaces, reverse the list, join.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Without split: reverse the whole character list, then reverse each word back, then compact the spaces with a write pointer. This is the in-place version asked in follow-ups.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** Split on spaces, reverse the list, join.

```python
def reverse_words(s):
    return " ".join(s.split()[::-1])
```

**Cost:** O(n) time, O(n) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Without split: reverse the whole character list, then reverse each word back, then compact the spaces with a write pointer. This is the in-place version asked in follow-ups.

```python
def reverse_words(s):
    a = list(s)

    def rev(l, r):
        while l < r:
            a[l], a[r] = a[r], a[l]
            l, r = l + 1, r - 1

    rev(0, len(a) - 1)
    n, i = len(a), 0
    while i < n:
        if a[i] == " ":
            i += 1
            continue
        j = i
        while j < n and a[j] != " ":
            j += 1
        rev(i, j - 1)
        i = j
    w = 0
    for r in range(n):
        if a[r] != " " or (w > 0 and a[w - 1] != " "):
            a[w] = a[r]
            w += 1
    if w > 0 and a[w - 1] == " ":
        w -= 1
    return "".join(a[:w])
```

**Cost:** O(n) time, O(1) extra for a mutable string space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: Split on spaces, reverse the list, join. That costs O(n) time and O(n) space."

"I can do better: Without split: reverse the whole character list, then reverse each word back, then compact the spaces with a write pointer. This is the in-place version asked in follow-ups. That's O(n) time and O(1) extra for a mutable string space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `s = 'the sky  blue'`

1. 1) Reverse every character
2. 2) Reverse the word at 0..3 back
3. 2) Reverse the word at 6..8 back
4. 2) Reverse the word at 10..12 back
5. 3) Squeeze extra spaces with a write pointer

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/reverse-words-in-a-string)

<a id="q20"></a>
### Q20. Trapping Rain Water

🔴 Stretch · *Opposite ends · running max*

Given bar heights, compute how much rain water is trapped between the bars.

**Examples**

- `height = [0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]` → `6`
- `height = [4, 2, 0, 3, 2, 5]` → `9`

In [ ]:
def trap(height):
    """
    Given bar heights, compute how much rain water is trapped between the bars.
    """
    # Write your code here
    pass


# ▶ Run this cell to test your answer
check(trap, [
    (([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1],), 6),
    (([4, 2, 0, 3, 2, 5],), 9),
])

<details>
<summary><b>💡 Hint 1: a way in</b></summary>

For each bar, scan left and right for the tallest walls; water above it is min(leftMax, rightMax) − height.

</details>

<details>
<summary><b>💡 Hint 2: the better idea</b></summary>

Move the pointer with the smaller running max. That side's water is fully decided by its own max, because the other side is known to be at least as tall.

</details>

<details>
<summary><b>✅ Answer: brute force</b></summary>

**Idea:** For each bar, scan left and right for the tallest walls; water above it is min(leftMax, rightMax) − height.

```python
def trap(height):
    water = 0
    for i in range(len(height)):
        left = max(height[:i + 1])
        right = max(height[i:])
        water += min(left, right) - height[i]
    return water
```

**Cost:** O(n²) time, O(1) space

</details>

<details>
<summary><b>✅ Answer: optimal</b></summary>

**Idea:** Move the pointer with the smaller running max. That side's water is fully decided by its own max, because the other side is known to be at least as tall.

```python
def trap(height):
    l, r = 0, len(height) - 1
    lmax = rmax = water = 0
    while l < r:
        if height[l] < height[r]:
            lmax = max(lmax, height[l])
            water += lmax - height[l]
            l += 1
        else:
            rmax = max(rmax, height[r])
            water += rmax - height[r]
            r -= 1
    return water
```

**Cost:** O(n) time, O(1) space

</details>

<details>
<summary><b>🎤 How to explain it out loud</b></summary>

"My first thought is: For each bar, scan left and right for the tallest walls; water above it is min(leftMax, rightMax) − height. That costs O(n²) time and O(1) space."

"I can do better: Move the pointer with the smaller running max. That side's water is fully decided by its own max, because the other side is known to be at least as tall. That's O(n) time and O(1) space."

</details>

<details>
<summary><b>🔎 Walk through an example</b></summary>

Input: `height = [3, 0, 2, 0, 4, 1, 2]`

1. Right side: water at 6 = rmax 2 − 2 = 0
2. Right side: water at 5 = rmax 2 − 1 = 1
3. Left side: water at 0 = lmax 3 − 3 = 0
4. Left side: water at 1 = lmax 3 − 0 = 3
5. Left side: water at 2 = lmax 3 − 2 = 1
6. Left side: water at 3 = lmax 3 − 0 = 3

</details>

[▶ Watch both answers run line by line](https://abpandey-dev.github.io/learn-python-from-scratch-to-advanced/handbook/#/q/two-pointers/trapping-rain-water)

---
# Part 4: Interview questions, out loud

Cover the answer, **say yours out loud**, then open it and compare. Aim for two or three clear sentences, then an example.

<details>
<summary><b>1. Two pointers or hash map for Two Sum?</b></summary>

Unsorted input that needs original indices → hash map, O(n) time, O(n) space. Sorted input (or indices not needed) → two pointers, O(n) time after sorting, O(1) space. Mention the trade-off.

</details>

<details>
<summary><b>2. Why must the array be sorted for converging pointers?</b></summary>

The decision to discard l or r depends on knowing every element between them is between a[l] and a[r]. Without sorted order, a too-small sum doesn't prove a[l] is useless.

</details>

<details>
<summary><b>3. How are duplicate triplets avoided in 3Sum?</b></summary>

Sort first. Skip the outer index when `nums[i] == nums[i−1]`, and after recording a triplet move `l` past all equal values (and r past equal values). Equal values at the same position would produce the same triplet.

</details>

<details>
<summary><b>4. What is the difference between two pointers and sliding window?</b></summary>

Sliding window is a special case of same-direction two pointers where the elements **between** the pointers form a contiguous window whose state (sum, counts) you maintain. Generic two pointers usually only care about the two pointed elements.

</details>

<details>
<summary><b>5. What does 'in place' mean?</b></summary>

The algorithm modifies the input structure itself and uses only O(1) extra memory (a few variables), rather than building a new array.

</details>


---
# Check yourself before moving on

You're ready for the next notebook when you can tick these **without looking**:

- [ ] I can explain: Why two pointers beat two loops
- [ ] I can explain: Opposite ends on a sorted array
- [ ] I can explain: Read and write pointers (in place)
- [ ] I can explain: One pointer per sequence
- [ ] I can explain: Harder two-pointer problems

**Revision drill.** Come back in a few days, set a 25-minute timer per question, and solve these from a blank cell:

- [ ] [Q16](#q16) 4Sum
- [ ] [Q17](#q17) 3Sum Closest
- [ ] [Q18](#q18) Boats to Save People
- [ ] [Q19](#q19) Reverse Words in a String
- [ ] [Q20](#q20) Trapping Rain Water

Spaced repetition beats re-reading: redoing a question after you've partly forgotten it is what makes it stick.